# Pair Cost/Loss Sensitivity Evaluation

This notebook adds the **second-order pair evaluation** step for the quantum pruning project.

The existing `cost_loss_table.csv` is **first order**:

```text
turn off one block → evaluate model → save one row
```

This notebook creates `pair_cost_loss_table.csv`, which is **second order**:

```text
turn off block i and block j together → evaluate model → save one row for the pair
```

No QUBO/Hamiltonian files are changed here. This notebook only generates the pair table. Later, the QUBO code can be adjusted to use the measured pair terms.


## Output idea

If the single table has 10 candidates, this notebook evaluates all unique pairs:

```text
10 candidates → 10 × 9 / 2 = 45 pair evaluations
```

For each pair we measure:

```text
how much accuracy we lose
how much F1 we lose
how much validation loss increases
how many parameters we save by pruning both blocks
```

It also calculates an optional interaction value:

```text
interaction_loss = pair_loss - single_block_i_loss - single_block_j_loss
```

This interaction value is useful later for QUBO second-order `Q_ij` terms, but it is calculated from the same 45 pair evaluations. It does **not** require extra model runs.


In [ ]:
# -----------------------------
# Configuration
# -----------------------------

MODEL_NAME = "convnext"
SPLIT = "test"
MAX_SAMPLES = 600
BATCH_SIZE = 16
NUM_WORKERS = 0

# Use the existing single-block table as the source of truth for candidate names.
# For all 10 original candidates, use: "cost_loss_table.csv"
# For the currently selected QUBO candidates, use: "qubo_outputs/cost_loss_table.csv"
SINGLE_COST_TABLE = "cost_loss_table.csv"

# New output file generated by this notebook.
OUTPUT_PAIR_TABLE = "pair_cost_loss_table.csv"

# Set to None for full evaluation.
# For quick debugging, set for example MAX_PAIRS = 3.
MAX_PAIRS = None

# If True and OUTPUT_PAIR_TABLE already exists, already evaluated pairs are skipped.
RESUME = True

# Save intermediate CSV every N evaluated pairs.
SAVE_EVERY = 5


In [ ]:
from __future__ import annotations

import itertools
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import v2
from tqdm.auto import tqdm

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download


In [ ]:
# -----------------------------
# Image preprocessing and dataset
# -----------------------------

def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    new_img = Image.new("RGB", target_size, (0, 0, 0))
    left = (target_size[0] - img.size[0]) // 2
    top = (target_size[1] - img.size[1]) // 2
    new_img.paste(img, (left, top))
    return new_img


def get_transform(model_name: str):
    if model_name == "convnext":
        return v2.Compose([
            v2.Lambda(lambda img: resize_and_pad(img)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
        ])

    if model_name == "swinv2":
        return transforms.Compose([
            transforms.Resize((192, 192)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ])

    raise ValueError(f"Unknown model_name: {model_name}")


class StreetViewSubset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y


def build_dataloader(model_name: str, split: str, max_samples: int, batch_size: int, num_workers: int):
    transform = get_transform(model_name)
    ds = load_dataset(
        "canada-guesser/Canadian-streetview-cities",
        split=f"{split}[:{max_samples}]",
    )
    wrapped = StreetViewSubset(ds, transform)
    return DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


In [ ]:
# -----------------------------
# Model loading and evaluation
# -----------------------------

def load_finetuned_model(model_name: str, device: torch.device):
    if model_name == "convnext":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="cnn_model/convnext_tiny_set_3_final.bin",
        )
        model = timm.create_model("convnext_tiny", pretrained=False, num_classes=15)
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict)

    elif model_name == "swinv2":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="vit_model/swinv2_base_window12_192_0_finetuned_canadian_streetview.bin",
        )
        model = timm.create_model("swinv2_base_window12_192", pretrained=False, num_classes=15)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False))

    else:
        raise ValueError(f"Unknown model: {model_name}")

    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total = 0
    preds: List[int] = []
    labels: List[int] = []

    for x, y in tqdm(loader, desc="evaluate", leave=False):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        if hasattr(logits, "logits"):
            logits = logits.logits
        loss = criterion(logits, y)

        total_loss += float(loss.item())
        total += int(y.numel())
        preds.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        labels.extend(y.detach().cpu().tolist())

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


In [ ]:
# -----------------------------
# Candidate wrapping / bypass logic
# -----------------------------

@dataclass
class Candidate:
    name: str
    module: nn.Module
    n_params: int


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


class CandidateWrapper(nn.Module):
    """
    bypass=False: normal block behavior.
    bypass=True : block is skipped and input x is returned unchanged.

    This is the same pruning simulation idea as in the current single-block sensitivity notebook.
    """
    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module
        self.bypass = False

    def forward(self, x, *args, **kwargs):
        if not self.bypass:
            return self.module(x, *args, **kwargs)
        return x


def _get_parent_and_child(model: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
    return parent, parts[-1]


def replace_module(model: nn.Module, module_name: str, new_module: nn.Module):
    parent, child_key = _get_parent_and_child(model, module_name)
    if child_key.isdigit():
        parent[int(child_key)] = new_module
    else:
        setattr(parent, child_key, new_module)


def build_candidates_from_single_table(model: nn.Module, single_df: pd.DataFrame) -> List[Candidate]:
    """
    The existing cost_loss_table.csv is the source of truth.
    We evaluate exactly the candidate blocks listed there.
    """
    if "candidate" not in single_df.columns:
        raise ValueError("single cost table must contain a 'candidate' column")

    modules = dict(model.named_modules())
    candidates: List[Candidate] = []

    for name in single_df["candidate"].astype(str).tolist():
        if name not in modules:
            raise KeyError(f"Candidate '{name}' from the CSV was not found in model.named_modules().")
        module = modules[name]
        candidates.append(Candidate(name=name, module=module, n_params=count_trainable_params(module)))

    return candidates


def wrap_candidates(model: nn.Module, candidates: List[Candidate]) -> Dict[str, CandidateWrapper]:
    wrappers: Dict[str, CandidateWrapper] = {}
    for cand in candidates:
        wrapper = CandidateWrapper(cand.module)
        replace_module(model, cand.name, wrapper)
        wrappers[cand.name] = wrapper
    return wrappers


def set_all_bypass(wrappers: Dict[str, CandidateWrapper], value: bool = False):
    for wrapper in wrappers.values():
        wrapper.bypass = value


def stage_from_candidate_name(name: str) -> str:
    m = re.match(r"^(stages|layers)\.(\d+)", name)
    return f"{m.group(1)}.{m.group(2)}" if m else "unknown"


## Check the existing single-block table

This is the table that already exists in the project. It has one row per block.


In [ ]:
single_df = pd.read_csv(SINGLE_COST_TABLE)
print(f"Single-block table: {SINGLE_COST_TABLE}")
print(f"Shape: {single_df.shape[0]} rows × {single_df.shape[1]} columns")
print("Columns:")
print(list(single_df.columns))

display(single_df.head())


### Current single-block table meaning

Each row means:

```text
one block is bypassed alone → model is evaluated → loss/accuracy/F1 drop is saved
```

Important columns:

| Column | Meaning |
|---|---|
| `candidate` | Block name, for example `stages.2.blocks.2` |
| `params` | Parameters inside this single block |
| `baseline_loss` | Original model loss before pruning |
| `pruned_loss` | Loss when this one block is bypassed |
| `loss_increase_raw` | `pruned_loss - baseline_loss` |
| `baseline_accuracy` | Original accuracy |
| `pruned_accuracy` | Accuracy after bypassing this one block |
| `accuracy_drop_raw` | `baseline_accuracy - pruned_accuracy` |
| `baseline_macro_f1` | Original macro F1 |
| `pruned_macro_f1` | Macro F1 after bypassing this one block |
| `f1_drop_raw` | `baseline_macro_f1 - pruned_macro_f1` |
| `L_i` | Normalized single-block loss penalty |
| `C_i` | Normalized single-block compression/parameter value |
| `pruning_attractiveness` | High compression with low loss looks attractive |


In [ ]:
# -----------------------------
# Pair evaluation
# -----------------------------

def run_pair_sensitivity(
    model: nn.Module,
    loader: DataLoader,
    candidates: List[Candidate],
    single_df: pd.DataFrame,
    device: torch.device,
    output_csv: str,
    max_pairs: Optional[int] = None,
    resume: bool = True,
    save_every: int = 5,
) -> pd.DataFrame:
    """
    Evaluates every unique pair (i, j), where i < j.

    For 10 candidates, this performs 45 model evaluations:
        C(10, 2) = 10 * 9 / 2 = 45

    Each evaluation bypasses two blocks at the same time.
    """
    output_path = Path(output_csv)

    # Map single-block results by candidate name.
    single_by_name = single_df.set_index("candidate").to_dict(orient="index")

    print("Wrapping candidate blocks...")
    wrappers = wrap_candidates(model, candidates)

    print("Evaluating baseline model once...")
    set_all_bypass(wrappers, False)
    baseline = evaluate(model, loader, device)
    print(
        f"Baseline: loss={baseline['loss']:.6f}, "
        f"acc={baseline['accuracy']:.4f}, "
        f"f1={baseline['macro_f1']:.4f}, "
        f"n={baseline['n_samples']}"
    )

    total_model_params = count_trainable_params(model)
    all_pairs = list(itertools.combinations(candidates, 2))
    if max_pairs is not None:
        all_pairs = all_pairs[:max_pairs]

    rows: List[Dict[str, Any]] = []
    done_pairs = set()
    if resume and output_path.exists():
        existing = pd.read_csv(output_path)
        rows = existing.to_dict(orient="records")
        for r in rows:
            done_pairs.add((r["candidate_i"], r["candidate_j"]))
        print(f"Resume enabled: loaded {len(done_pairs)} completed pairs from {output_csv}")

    for pair_idx, (cand_i, cand_j) in enumerate(tqdm(all_pairs, desc="pair evaluations"), start=1):
        pair_key = (cand_i.name, cand_j.name)
        if pair_key in done_pairs:
            continue

        print(f"
Pair {pair_idx}/{len(all_pairs)}: {cand_i.name} + {cand_j.name}")

        # Turn off exactly two blocks for this evaluation.
        set_all_bypass(wrappers, False)
        wrappers[cand_i.name].bypass = True
        wrappers[cand_j.name].bypass = True

        pair_metrics = evaluate(model, loader, device)

        # Restore normal model behavior before the next pair.
        set_all_bypass(wrappers, False)

        row_i = single_by_name[cand_i.name]
        row_j = single_by_name[cand_j.name]

        pair_loss_increase = pair_metrics["loss"] - baseline["loss"]
        pair_accuracy_drop = baseline["accuracy"] - pair_metrics["accuracy"]
        pair_f1_drop = baseline["macro_f1"] - pair_metrics["macro_f1"]

        single_i_loss = float(row_i.get("loss_increase_raw", 0.0))
        single_j_loss = float(row_j.get("loss_increase_raw", 0.0))
        expected_additive_loss = single_i_loss + single_j_loss
        interaction_loss = pair_loss_increase - expected_additive_loss

        single_i_acc_drop = float(row_i.get("accuracy_drop_raw", 0.0))
        single_j_acc_drop = float(row_j.get("accuracy_drop_raw", 0.0))
        expected_additive_acc_drop = single_i_acc_drop + single_j_acc_drop
        interaction_acc_drop = pair_accuracy_drop - expected_additive_acc_drop

        single_i_f1_drop = float(row_i.get("f1_drop_raw", 0.0))
        single_j_f1_drop = float(row_j.get("f1_drop_raw", 0.0))
        expected_additive_f1_drop = single_i_f1_drop + single_j_f1_drop
        interaction_f1_drop = pair_f1_drop - expected_additive_f1_drop

        pair_params = int(cand_i.n_params + cand_j.n_params)
        stage_i = stage_from_candidate_name(cand_i.name)
        stage_j = stage_from_candidate_name(cand_j.name)

        row = {
            "pair_id": f"{cand_i.name}__PLUS__{cand_j.name}",
            "candidate_i": cand_i.name,
            "candidate_j": cand_j.name,
            "stage_i": stage_i,
            "stage_j": stage_j,
            "same_stage": stage_i == stage_j,
            "params_i": int(cand_i.n_params),
            "params_j": int(cand_j.n_params),
            "pair_params": pair_params,
            "total_model_params": int(total_model_params),
            "pair_param_reduction_pct": pair_params / max(total_model_params, 1),
            "baseline_loss": baseline["loss"],
            "pair_pruned_loss": pair_metrics["loss"],
            "pair_loss_increase_raw": pair_loss_increase,
            "single_i_loss_increase_raw": single_i_loss,
            "single_j_loss_increase_raw": single_j_loss,
            "expected_additive_loss_increase_raw": expected_additive_loss,
            "interaction_loss_raw": interaction_loss,
            "interaction_loss_positive_raw": max(interaction_loss, 0.0),
            "baseline_accuracy": baseline["accuracy"],
            "pair_pruned_accuracy": pair_metrics["accuracy"],
            "pair_accuracy_drop_raw": pair_accuracy_drop,
            "single_i_accuracy_drop_raw": single_i_acc_drop,
            "single_j_accuracy_drop_raw": single_j_acc_drop,
            "expected_additive_accuracy_drop_raw": expected_additive_acc_drop,
            "interaction_accuracy_drop_raw": interaction_acc_drop,
            "baseline_macro_f1": baseline["macro_f1"],
            "pair_pruned_macro_f1": pair_metrics["macro_f1"],
            "pair_f1_drop_raw": pair_f1_drop,
            "single_i_f1_drop_raw": single_i_f1_drop,
            "single_j_f1_drop_raw": single_j_f1_drop,
            "expected_additive_f1_drop_raw": expected_additive_f1_drop,
            "interaction_f1_drop_raw": interaction_f1_drop,
            "n_samples": pair_metrics["n_samples"],
            # Raw values intended for later QUBO use.
            "L_ij_raw": max(pair_loss_increase, 0.0),
            "C_ij_raw": float(pair_params),
            "Q_ij_raw": max(interaction_loss, 0.0),
        }
        rows.append(row)

        print(
            f"  pair loss={pair_metrics['loss']:.6f}, acc={pair_metrics['accuracy']:.4f}, f1={pair_metrics['macro_f1']:.4f}
"
            f"  pair drops: Δloss={pair_loss_increase:.6f}, Δacc={pair_accuracy_drop:.4f}, Δf1={pair_f1_drop:.4f}
"
            f"  saved params={pair_params:,} ({row['pair_param_reduction_pct']:.2%} of trainable model params)
"
            f"  interaction loss={interaction_loss:.6f}"
        )

        if save_every and len(rows) % save_every == 0:
            pd.DataFrame(rows).to_csv(output_path, index=False)
            print(f"  intermediate save → {output_path}")

    pair_df = pd.DataFrame(rows)

    if not pair_df.empty:
        max_L = float(pair_df["L_ij_raw"].max())
        max_C = float(pair_df["C_ij_raw"].max())
        max_Q = float(pair_df["Q_ij_raw"].max())

        pair_df["L_ij"] = pair_df["L_ij_raw"] / max_L if max_L > 0 else 0.0
        pair_df["C_ij"] = pair_df["C_ij_raw"] / max_C if max_C > 0 else 0.0
        pair_df["Q_ij"] = pair_df["Q_ij_raw"] / max_Q if max_Q > 0 else 0.0
        pair_df["pair_pruning_attractiveness"] = pair_df["C_ij"] / (pair_df["L_ij"] + 1e-6)

    pair_df.to_csv(output_path, index=False)
    print(f"
Saved pair table to: {output_path}")
    print(f"Rows: {len(pair_df)}")
    return pair_df


## Run the pair evaluation

This cell performs the pair evaluations.

For the default `cost_loss_table.csv` with 10 candidates, it should run 45 pair evaluations.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print("Loading model...")
model = load_finetuned_model(MODEL_NAME, device)

print("Building dataloader...")
loader = build_dataloader(
    model_name=MODEL_NAME,
    split=SPLIT,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

candidates = build_candidates_from_single_table(model, single_df)
print(f"Candidates from {SINGLE_COST_TABLE}: {len(candidates)}")
print(f"Expected pair evaluations: {len(candidates) * (len(candidates) - 1) // 2}")

pair_df = run_pair_sensitivity(
    model=model,
    loader=loader,
    candidates=candidates,
    single_df=single_df,
    device=device,
    output_csv=OUTPUT_PAIR_TABLE,
    max_pairs=MAX_PAIRS,
    resume=RESUME,
    save_every=SAVE_EVERY,
)

display(pair_df.head())


## New pair table meaning

The new output file is:

```text
pair_cost_loss_table.csv
```

Each row means:

```text
candidate_i + candidate_j are bypassed together → model is evaluated → pair result is saved
```

Important columns:

| Column | Meaning |
|---|---|
| `candidate_i` | First block in the pair |
| `candidate_j` | Second block in the pair |
| `same_stage` | Whether both blocks are in the same model stage |
| `params_i`, `params_j` | Parameters inside each block |
| `pair_params` | Parameters saved if both blocks are pruned |
| `pair_param_reduction_pct` | Estimated pair parameter reduction compared with trainable model params |
| `baseline_loss` | Original model loss |
| `pair_pruned_loss` | Loss when both blocks are bypassed together |
| `pair_loss_increase_raw` | `pair_pruned_loss - baseline_loss` |
| `pair_pruned_accuracy` | Accuracy when both blocks are bypassed together |
| `pair_accuracy_drop_raw` | `baseline_accuracy - pair_pruned_accuracy` |
| `pair_pruned_macro_f1` | Macro F1 when both blocks are bypassed together |
| `pair_f1_drop_raw` | `baseline_macro_f1 - pair_pruned_macro_f1` |
| `expected_additive_loss_increase_raw` | Single-block loss of i + single-block loss of j |
| `interaction_loss_raw` | Extra pair damage beyond the sum of individual damages |
| `L_ij_raw` | Pair loss penalty for later QUBO use |
| `C_ij_raw` | Pair compression/parameter value for later QUBO use |
| `Q_ij_raw` | Positive pair interaction penalty for later QUBO use |

The most direct columns for your current goal are:

```text
candidate_i
candidate_j
pair_params
pair_param_reduction_pct
pair_loss_increase_raw
pair_accuracy_drop_raw
pair_f1_drop_raw
```

These answer exactly:

```text
which two blocks did we remove?
how much did we save?
how much accuracy/loss/F1 did we lose?
```


In [ ]:
# Clean view of the most important pair columns.
important_cols = [
    "candidate_i", "candidate_j",
    "pair_params", "pair_param_reduction_pct",
    "pair_loss_increase_raw", "pair_accuracy_drop_raw", "pair_f1_drop_raw",
    "interaction_loss_raw",
]

if Path(OUTPUT_PAIR_TABLE).exists():
    pair_preview = pd.read_csv(OUTPUT_PAIR_TABLE)
    display(pair_preview[important_cols].head(10))
else:
    print(f"{OUTPUT_PAIR_TABLE} does not exist yet. Run the previous cell first.")


## What changed in the project?

Files changed:

```text
none
```

Files added:

```text
pair_cost_loss_sensitivity.ipynb
```

Files generated when you run the notebook:

```text
pair_cost_loss_table.csv
```

The rest of the pipeline is not changed yet. Later we can modify `qubo_hamiltonian.py` or its notebook so that it reads `pair_cost_loss_table.csv` and uses the measured pair penalties as second-order QUBO terms.
